In [1]:
import numpy as np
import pandas as pd
from scipy.stats import linregress, pearsonr

from armored.models import *
from armored.preprocessing import *

import itertools

from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeRegressor
from sklearn import tree

In [2]:
# data with initial and end point measurements
df_exp0 = pd.read_csv("data/exp0/exp0_metabolites.csv")
df_exp1 = pd.read_csv("data/exp1/exp1_metabolites.csv")
df_exp2 = pd.read_csv("data/exp2/exp2_metabolites.csv")
df_exp3 = pd.read_csv("data/exp3/exp3_metabolites.csv")
df_exp4a = pd.read_csv("data/exp4/exp4_metabolites_best_reps.csv")

exp_names_reps = [e+"-rep" for e in df_exp4a.Experiments.values]
df_exp4a['Experiments'] = exp_names_reps

df_exp4b = pd.read_csv("data/exp4/exp4_metabolites_new_best.csv")
df_exp4c = pd.read_csv("data/exp4/exp4_metabolites_new_worst.csv")
df = pd.concat((df_exp0, df_exp1, df_exp2, df_exp3, df_exp4a, df_exp4b, df_exp4c))

# define variable names
species = ['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs', 'CHabs',
           'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs', 'RIabs']
metabolites = ['pH', 'Lactate', 'Butyrate', 'Acetate']
controls = ['AcGum', 'ArGal', 'Inulin', 'Pectin', 'Starch', 'Xylan']

# concatenate all observed and all system variables 
observed = np.concatenate((np.array(species), np.array(metabolites)))
system_variables = np.concatenate((np.array(species), np.array(metabolites), np.array(controls)))
system_variables

# # average replicates
# df_fmt_mean = []
# for exp_name, df_exp in df.groupby("Experiments"):
#     df_groups = df_exp.groupby("Time")
#     df_avg = df_groups[system_variables].mean().reset_index()
#     df_avg.insert(0, "Experiments", [exp_name]*df_avg.shape[0])
#     df_fmt_mean.append(df_avg)
# df = pd.concat(df_fmt_mean)

array(['ACabs', 'BAabs', 'BHabs', 'BLabs', 'BUabs', 'CAabs', 'CCabs',
       'CHabs', 'DFabs', 'ELabs', 'ERabs', 'FPabs', 'PCabs', 'PJabs',
       'RIabs', 'pH', 'Lactate', 'Butyrate', 'Acetate', 'AcGum', 'ArGal',
       'Inulin', 'Pectin', 'Starch', 'Xylan'], dtype='<U8')

In [3]:
# log that ignores zeros
def zlog(x):
    x[x <= 0] = 1
    return np.log(x)

# shannon diversity
def shannon(y):
    y = np.clip(y, 0, np.inf)
    if np.nansum(y) > 0:
        y_rel = y / np.nansum(y)
        return np.nansum(-zlog(y_rel)*y_rel)
    else:
        return np.nan

# determine best previously observed values of objectives
b_max = 24.206144
d_max = 2.2569
v_max = 0.05319

# define objective 
def objective(y):
    
    # y is measured exp data [n_time, n_species + n_metabolites]
    if len(y)>2:
        # endpoint shannon diversity
        diversity = shannon(y[-1, :len(species)])

        # variance in species abundances in last two passages
        if np.any(np.isnan(y[-2:, :len(species)])):
            instability = np.nan
        else:
            species_stdv = np.std(y[-2:, :len(species)], 0)
            instability  = np.where(species_stdv>0, species_stdv, 0).mean() 

        # endpoint butyrate production 
        butyrate =  y[-1, -2]   

        # scaled objective 
        scaled_obj = diversity / d_max - instability / v_max + butyrate / b_max

        return diversity, instability, butyrate, scaled_obj
    
    else:
        return np.nan, np.nan, np.nan, np.nan

# function handle evaluates objective in batches
# objective_batch = vmap(objective)
def objective_batch(y):
    div_batch = np.zeros(len(y))
    inst_batch = np.zeros(len(y))
    buty_batch = np.zeros(len(y))
    obj_batch = np.zeros(len(y))
    for i, yi in enumerate(y):
        div, inst, buty, obj = objective(yi)
        div_batch[i] = div
        inst_batch[i] = inst
        buty_batch[i] = buty
        obj_batch[i] = obj
    return div_batch, inst_batch, buty_batch, obj_batch

In [4]:
# compute measured objectives 
exp_objs = {}
for exp_name, df_exp in df.groupby("Experiments"):
    exp_objs[exp_name] = objective(df_exp[species+metabolites].values)

In [5]:
# scale data 
scaler = MinMaxScaler(observed, system_variables)
scaler.fit(df)
df_scaled = scaler.transform(df.copy())

# format data into matrix [n_samples, n_timepoints, dt+n_outputs+n_controls]
data = format_data(df.copy(), species, metabolites, controls, observed=observed)
data_scaled = format_data(df_scaled, species, metabolites, controls, observed=observed)

# instantiate model
brnn = miRNN(n_species=len(species), n_metabolites=len(metabolites), n_controls=len(controls), n_hidden=32)
# fit model
brnn.fit(data_scaled, evd_tol=1e-3)

Total measurements: 27433, Number of parameters: 2515, Initial regularization: 0.00e+00
Loss: 1361.368, Residuals: -0.00458
Loss: 1276.159, Residuals: -0.00132
Loss: 1245.785, Residuals: 0.00123
Loss: 1215.626, Residuals: 0.00145
Loss: 1101.710, Residuals: 0.00423
Loss: 1074.427, Residuals: -0.00225
Loss: 1025.881, Residuals: -0.00054
Loss: 935.734, Residuals: -0.00059
Loss: 899.029, Residuals: -0.00067
Loss: 841.014, Residuals: -0.00068
Loss: 797.382, Residuals: -0.00104
Loss: 793.332, Residuals: -0.00110
Loss: 785.543, Residuals: -0.00103
Loss: 771.632, Residuals: -0.00089
Loss: 745.389, Residuals: -0.00141
Loss: 697.096, Residuals: -0.00176
Loss: 649.786, Residuals: -0.00136
Loss: 638.658, Residuals: -0.00103
Loss: 617.765, Residuals: -0.00081
Loss: 584.053, Residuals: -0.00066
Loss: 582.449, Residuals: -0.00073
Loss: 579.644, Residuals: -0.00110
Loss: 574.277, Residuals: -0.00112
Loss: 564.315, Residuals: -0.00108
Loss: 546.914, Residuals: -0.00127
Loss: 530.030, Residuals: -0.0011

In [6]:
# save predictions to dataframe
pred_dfs = []
for T, X, U, Y, exp_names in data: 
    
    # keep initial condition
    P = np.array(brnn.predict_point(X, U))
    
    # unscale predictions
    for eval_time in scaler.eval_times:
        t_inds = T == eval_time
        P[t_inds] *= (scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"])
        P[t_inds] += scaler.scale_dict_obs[f"{eval_time} min"]
    
    # get shapes
    batch_size, n_time, n_out = P.shape
    
    # predicted objective
    div_pred, inst_pred, but_pred, obj_pred = objective_batch(P)
    
    # measured objective
    div_meas = []
    inst_meas = []
    but_meas = []
    obj_meas = []
    for exp_name in exp_names:
        div_, inst_, but_, obj_ = exp_objs[exp_name]
        div_meas.append(div_)
        inst_meas.append(inst_)
        but_meas.append(but_)
        obj_meas.append(obj_)
    div_meas = np.array(div_meas)
    inst_meas = np.array(inst_meas)
    but_meas = np.array(but_meas)
    obj_meas = np.array(obj_meas)
    
    # reshape arrays
    T = np.reshape(T, batch_size*n_time)
    U = np.reshape(U, [batch_size*n_time, U.shape[-1]])
    Y = np.reshape(Y, [batch_size*n_time, n_out])
    P = np.reshape(P, [batch_size*n_time, n_out])
    exp_names_array = np.reshape(np.array([np.tile(item, n_time) for item in exp_names]), n_time * len(exp_names))
    
    # names of variables
    outputs = species+metabolites
    outputs_meas = [o + " meas" for o in outputs]
    outputs_pred = [o + " pred" for o in outputs]
    
    # save to df
    pred_df = pd.DataFrame()
    pred_df['Experiments'] = exp_names_array
    pred_df['Time'] = T
    
    pred_df['Diversity meas'] = np.repeat(div_meas, n_time)
    pred_df['Instability meas'] = np.repeat(inst_meas, n_time)
    pred_df['Butyrate meas'] = np.repeat(but_meas, n_time)
    pred_df['Objective meas'] = np.repeat(obj_meas, n_time)
    
    pred_df['Diversity pred'] = np.repeat(div_pred, n_time)
    pred_df['Instability pred'] = np.repeat(inst_pred, n_time)
    pred_df['Butyrate pred'] = np.repeat(but_pred, n_time)
    pred_df['Objective pred'] = np.repeat(obj_pred, n_time)
    
    pred_df[outputs_meas] = Y
    pred_df[outputs_pred] = P
    pred_df[controls] = U
    pred_dfs.append(pred_df)
    
pred_df = pd.concat(pred_dfs)
pred_df.to_csv("space/sampled_space.csv", index=False)

In [7]:
np.nanmax(pred_df['Objective meas'].values)

1.379517274801405

In [8]:
# full factorial of both species and controls 
n_species = len(species)
n_fibers  = len(controls)

# create matrix of all possible communities
Slist = [np.reshape(np.array(i), (1, n_species)) for i in itertools.product([0, 1], repeat = n_species)]
# remove all zeros community
S = np.array(np.concatenate(Slist)[1:], float)

# create matrix of all possible fiber combinations
Flist = [np.reshape(np.array(i), (1, n_fibers)) for i in itertools.product([0, 1], repeat = n_fibers)]
F = np.array(np.concatenate(Flist), float)

# matrix of species and fiber indeces 
M = np.array(np.zeros([S.shape[0]*F.shape[0], 2]), int)
k = 0 
for i in range(S.shape[0]):
    for j in range(F.shape[0]):
        M[k, 0] = int(i)
        M[k, 1] = int(j)
        k += 1

# function to pull sample data 
def gen_exp_cond(k):
    s_ind, f_ind = M[k]
    return S[s_ind], F[f_ind]

# function to generate informative name of experimental condition
def gen_exp_name(Si, Fi):
    exp_name = ""
    for i,si in enumerate(Si):
        if si > 0:
            exp_name += species[i].split("abs")[0] + "-"
    for i,fi in enumerate(Fi):
        if fi > 0:
            exp_name += controls[i] + "-"
    return exp_name[:-1]

In [9]:
def format_design_data(S, F, t_eval, scaler, batch_size=512):

    # data is a list of tuples (T, X, U, Y, names) where each tuple corresponds to a batch
    data = []
    
    # total number of samples
    n_samples = S.shape[0] * F.shape[0]
    k = 0 
    
    # number of evaluation times
    n_eval = len(t_eval)

    # divide data into batches
    for batch_inds in tqdm(np.array_split(np.arange(n_samples), np.ceil(n_samples / batch_size))):

        # initialize data matrices 
        T = np.empty([len(batch_inds), n_eval])
        T[:] = t_eval
        X = np.empty([len(batch_inds), S.shape[-1]+len(metabolites)])
        X[:] = np.nan
        U = np.empty([len(batch_inds), n_eval, F.shape[-1]])
        U[:] = np.nan

        # keep track of experiment names
        names = []
        for i, batch_ind in enumerate(batch_inds):

            # pull sample data 
            Si, Fi = gen_exp_cond(k)

            # keep track of experiment names
            names.append(gen_exp_name(Si, Fi))

            # store initial condition data
            X[i, :len(Si)] = .000667 * Si
            
            # initial pH and metabolites
            X[i, len(Si):] = np.array([6.7, 0., 0., 0.])
            
            # scale observed variables 
            X[i] = (X[i] - scaler.scale_dict_obs["0 min"]) / scaler.scale_dict_obs["0 max"]
            
            # store controls (already scaled)
            if sum(Fi)>0:
                U[i] =  Fi / sum(Fi)
            else:
                U[i] = Fi
            
            # update counter
            k += 1

        data.append((T, X, U, names))

    return data

In [10]:
# format and scale data based on training data
design_data = format_design_data(S, F, np.array([0, 1, 2, 3]), scaler, batch_size=1024)

100%|███████████████████████████████████████| 2048/2048 [00:41<00:00, 49.73it/s]


In [11]:
# save predictions to dataframe
pred_dfs = []
# for T, X, U, Y, exp_names in data_scaled: 
for T, X, U, exp_names in design_data:
    
    # keep initial condition
    preds = np.array(brnn.predict_point(X, U))
    
    # unscale predictions
    for eval_time in scaler.eval_times:
        t_inds = T == eval_time
        preds[t_inds] *= (scaler.scale_dict_obs[f"{eval_time} max"] - scaler.scale_dict_obs[f"{eval_time} min"])
        preds[t_inds] += scaler.scale_dict_obs[f"{eval_time} min"]
    
    # get shapes
    batch_size, n_time, n_out = preds.shape
    
    # reshape arrays
    T = np.reshape(T, batch_size*n_time)
    U = np.reshape(U, [batch_size*n_time, U.shape[-1]])
    Y = np.reshape(preds, [batch_size*n_time, n_out])
    exp_names_array = np.reshape(np.array([np.tile(item, n_time) for item in exp_names]), n_time * len(exp_names))
    
    # save to df
    pred_df = pd.DataFrame()
    pred_df['Experiments'] = exp_names_array
    pred_df['Time'] = T
    pred_df[species+metabolites] = Y
    pred_df[controls] = U
    pred_dfs.append(pred_df)
    
pred_df = pd.concat(pred_dfs)

In [13]:
# # all predicted variables
# y = pred_df[observed].values

# # reshape into [n_samples, n_passages, n_variables]
# y_batch = np.reshape(y, [y.shape[0]//4, 4, y.shape[-1]])

# # evaluate obective of every sample
# obj_batch = objective_batch(y_batch)

# # store objective for all time points
# pred_df['Objective'] = np.repeat(obj_batch, 4)

In [ ]:
# save to csv
# pred_df.to_csv("space/space.csv", index=False)